# 🔍 Phase 1A-EDA: Deep Exploratory Data Analysis & Cleaning
**FMA Small Dataset — ก่อน Pre-extract Features**

Notebook นี้จะวิเคราะห์ dataset อย่างละเอียดเพื่อหา:
1. **Audio Duration Distribution** — เพลงสั้น/ยาวผิดปกติ
2. **Class Imbalance** — genres ที่มี samples ไม่สมดุล
3. **Silent / Corrupted Track Detection** — เพลงที่ RMS ≈ 0 หรือ load ไม่ได้
4. **Feature Correlation & Discriminability** — features ตัวไหนสำคัญ/ซ้ำซ้อน
5. **Mel Spectrogram Statistics** — วิเคราะห์ dB distribution สำหรับ normalization
6. **Per-Genre Audio Characteristics** — แต่ละ genre เสียงต่างกันยังไง
7. **Cleaning Summary & Updated Skip-list** — สรุปสิ่งที่ต้องแก้

ผลลัพธ์จะถูกบันทึกเป็น `cleaned_metadata.csv` สำหรับใช้ใน Phase 1B


## 0. Setup & Load Phase 1A Pipeline

In [1]:
import os
import time
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

warnings.filterwarnings('ignore')

# ============================================================
# PATHS & CONFIG (เหมือน Phase 1A)
# ============================================================
FMA_AUDIO_DIR = Path(r"D:\patt\project\pattern-music\FMA_Data\fma_small\fma_small")
FMA_METADATA_CSV = Path(r"D:\patt\project\pattern-music\FMA_Data\fma_metadata\fma_metadata\tracks.csv")

SAMPLE_RATE = 22050
CHUNK_DURATION = 3.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
TARGET_TIME_FRAMES = 130

CORRUPTED_TRACK_IDS = {98565, 98567, 98569, 99134, 108925, 133297, 143992}

GENRE_LABELS = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop',
                'Instrumental', 'International', 'Pop', 'Rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRE_LABELS)}
NUM_CLASSES = len(GENRE_LABELS)

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Setup complete")


✓ Setup complete


## 1. Load Metadata & Basic Stats

In [2]:
# Load tracks.csv (multi-level headers)
tracks = pd.read_csv(FMA_METADATA_CSV, index_col=0, header=[0, 1])

# Filter fma_small
small_mask = tracks[('set', 'subset')] == 'small'
df = tracks.loc[small_mask, [('track', 'genre_top'), ('set', 'split'), ('track', 'duration')]].copy()
df.columns = ['genre_top', 'split', 'duration_metadata']
df.index.name = 'track_id'

# Drop NaN genres
df = df.dropna(subset=['genre_top'])

# Drop known corrupted
df = df.drop(index=list(CORRUPTED_TRACK_IDS.intersection(set(df.index))), errors='ignore')

# Add genre_idx
df['genre_idx'] = df['genre_top'].map(GENRE_TO_IDX)

print(f"Total tracks after initial cleaning: {len(df)}")
print(f"\nSplit distribution:")
print(df['split'].value_counts().to_string())
print(f"\nGenre distribution:")
print(df['genre_top'].value_counts().sort_index().to_string())
print(f"\nDuration from metadata (seconds):")
print(df['duration_metadata'].describe().to_string())


Total tracks after initial cleaning: 7994

Split distribution:
split
training      6394
validation     800
test           800

Genre distribution:
genre_top
Electronic        999
Experimental      999
Folk             1000
Hip-Hop           997
Instrumental     1000
International    1000
Pop              1000
Rock              999

Duration from metadata (seconds):
count    7994.000000
mean      228.814111
std        98.330964
min        60.000000
25%       161.000000
50%       215.000000
75%       279.000000
max       600.000000


## 2. 🎵 Audio Duration & File Size Analysis

สแกนทุก audio file เพื่อวัด:
- **จริงๆ แล้วเพลงยาวกี่วินาที** (จาก actual audio, ไม่ใช่ metadata)
- **File size** — ไฟล์เล็กผิดปกติอาจ corrupted
- **Load success/failure** — track ไหน load ไม่ได้

⚠️ Cell นี้ใช้เวลา ~10-20 นาที (สแกน 8000 tracks)


In [ ]:
# สแกนทุก track — เก็บ duration, file_size, load_success
scan_results = []

print("Scanning all audio files...")
start_time = time.time()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scanning"):
    track_id = idx
    tid_str = f"{track_id:06d}"
    audio_path = FMA_AUDIO_DIR / tid_str[:3] / f"{tid_str}.mp3"

    result = {
        'track_id': track_id,
        'genre_top': row['genre_top'],
        'split': row['split'],
        'genre_idx': row['genre_idx'],
    }

    if not audio_path.exists():
        result.update({'file_exists': False, 'file_size_kb': 0,
                       'duration_actual': 0, 'load_success': False,
                       'rms_mean': 0, 'rms_max': 0, 'error': 'file_not_found'})
    else:
        result['file_exists'] = True
        result['file_size_kb'] = audio_path.stat().st_size / 1024

        try:
            audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
            duration = len(audio) / sr
            rms = librosa.feature.rms(y=audio, hop_length=HOP_LENGTH)[0]

            result.update({
                'duration_actual': duration,
                'load_success': True,
                'rms_mean': float(rms.mean()),
                'rms_max': float(rms.max()),
                'num_samples': len(audio),
                'error': None,
            })
        except Exception as e:
            result.update({
                'duration_actual': 0,
                'load_success': False,
                'rms_mean': 0, 'rms_max': 0,
                'error': str(e),
            })

    scan_results.append(result)

elapsed = time.time() - start_time
scan_df = pd.DataFrame(scan_results)
print(f"\n✓ Scan complete in {elapsed/60:.1f} minutes")
print(f"  Total tracks scanned: {len(scan_df)}")
print(f"  Load success: {scan_df['load_success'].sum()}")
print(f"  Load failed:  {(~scan_df['load_success']).sum()}")


Scanning all audio files...


Scanning:   0%|          | 0/7994 [00:00<?, ?it/s]

### 2.1 Duration Distribution

In [ ]:
successful = scan_df[scan_df['load_success']].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 2.1a — Duration histogram
axes[0].hist(successful['duration_actual'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(x=CHUNK_DURATION, color='red', linestyle='--', linewidth=2, label=f'Chunk size ({CHUNK_DURATION}s)')
axes[0].axvline(x=successful['duration_actual'].median(), color='orange', linestyle='--', label=f'Median ({successful["duration_actual"].median():.0f}s)')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Audio Duration Distribution')
axes[0].legend()

# 2.1b — Short tracks (< 10s)
short_tracks = successful[successful['duration_actual'] < 10]
axes[1].hist(short_tracks['duration_actual'], bins=30, color='#FF5722', edgecolor='white', alpha=0.8)
axes[1].axvline(x=CHUNK_DURATION, color='red', linestyle='--', linewidth=2, label=f'Chunk size ({CHUNK_DURATION}s)')
axes[1].set_xlabel('Duration (seconds)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Short Tracks (< 10s): {len(short_tracks)} tracks')
axes[1].legend()

# 2.1c — Duration by genre
genre_durations = successful.groupby('genre_top')['duration_actual'].median().sort_values()
genre_durations.plot(kind='barh', ax=axes[2], color='#4CAF50', edgecolor='white')
axes[2].set_xlabel('Median Duration (seconds)')
axes[2].set_title('Median Duration by Genre')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Stats
print(f"Duration statistics:")
print(f"  Min:    {successful['duration_actual'].min():.1f}s")
print(f"  Max:    {successful['duration_actual'].max():.1f}s")
print(f"  Mean:   {successful['duration_actual'].mean():.1f}s")
print(f"  Median: {successful['duration_actual'].median():.1f}s")
print(f"\n  Tracks shorter than {CHUNK_DURATION}s (will be padded): {len(successful[successful['duration_actual'] < CHUNK_DURATION])}")
print(f"  Tracks shorter than 1s (problematic): {len(successful[successful['duration_actual'] < 1])}")


### 2.2 File Size Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# File size distribution
axes[0].hist(successful['file_size_kb'], bins=50, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[0].axvline(x=10, color='red', linestyle='--', label='10 KB threshold')
axes[0].set_xlabel('File Size (KB)')
axes[0].set_ylabel('Count')
axes[0].set_title('File Size Distribution')
axes[0].legend()

# Tiny files
tiny_files = successful[successful['file_size_kb'] < 10]
if len(tiny_files) > 0:
    axes[1].barh(range(len(tiny_files)), tiny_files['file_size_kb'].values)
    axes[1].set_yticks(range(len(tiny_files)))
    axes[1].set_yticklabels([f"Track {tid}" for tid in tiny_files['track_id'].values])
    axes[1].set_xlabel('File Size (KB)')
    axes[1].set_title(f'Suspiciously Small Files (< 10 KB): {len(tiny_files)}')
else:
    axes[1].text(0.5, 0.5, 'No tiny files found ✓', ha='center', va='center', fontsize=14)
    axes[1].set_title('Tiny Files Check')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_file_size.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"File size stats:")
print(f"  Min: {successful['file_size_kb'].min():.1f} KB")
print(f"  Max: {successful['file_size_kb'].max():.1f} KB")
print(f"  Mean: {successful['file_size_kb'].mean():.1f} KB")


## 3. 🔇 Silent & Near-Silent Track Detection

ตรวจเพลงที่ RMS energy ต่ำผิดปกติ — เพลงเงียบจะทำให้ features เป็น garbage
ซึ่งจะ confuse โมเดลตอน training


In [ ]:
# RMS distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RMS mean distribution
axes[0].hist(successful['rms_mean'], bins=50, color='#00BCD4', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('RMS Mean Energy')
axes[0].set_ylabel('Count')
axes[0].set_title('RMS Mean Energy Distribution')
axes[0].set_yscale('log')

# RMS by genre
rms_by_genre = successful.groupby('genre_top')['rms_mean'].agg(['mean', 'std']).sort_values('mean')
rms_by_genre['mean'].plot(kind='barh', ax=axes[1], xerr=rms_by_genre['std'],
                           color='#FF9800', edgecolor='white', capsize=3)
axes[1].set_xlabel('Mean RMS Energy')
axes[1].set_title('RMS Energy by Genre')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_rms_energy.png', dpi=150, bbox_inches='tight')
plt.show()

# Detect silent tracks (RMS < threshold)
RMS_SILENCE_THRESHOLD = 0.001
silent_tracks = successful[successful['rms_mean'] < RMS_SILENCE_THRESHOLD]

print(f"\nSilent/Near-silent tracks (RMS mean < {RMS_SILENCE_THRESHOLD}):")
print(f"  Count: {len(silent_tracks)}")
if len(silent_tracks) > 0:
    for _, row in silent_tracks.iterrows():
        print(f"  Track {row['track_id']:6d} | RMS: {row['rms_mean']:.6f} | Genre: {row['genre_top']} | Duration: {row['duration_actual']:.1f}s")

# Near-silent (very quiet but not completely silent)
NEAR_SILENCE_THRESHOLD = 0.005
near_silent = successful[(successful['rms_mean'] >= RMS_SILENCE_THRESHOLD) &
                          (successful['rms_mean'] < NEAR_SILENCE_THRESHOLD)]
print(f"\nNear-silent tracks ({RMS_SILENCE_THRESHOLD} ≤ RMS < {NEAR_SILENCE_THRESHOLD}):")
print(f"  Count: {len(near_silent)}")
if len(near_silent) > 0:
    for _, row in near_silent.head(10).iterrows():
        print(f"  Track {row['track_id']:6d} | RMS: {row['rms_mean']:.6f} | Genre: {row['genre_top']}")


## 4. ⚖️ Class Imbalance Analysis

ดู genre distribution ละเอียด + คำนวณ class weights สำหรับ training


In [ ]:
# Genre distribution per split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, split in enumerate(['training', 'validation', 'test']):
    split_df = successful[successful['split'] == split]
    genre_counts = split_df['genre_top'].value_counts().sort_index()

    colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
    genre_counts.plot(kind='bar', ax=axes[i], color=colors, edgecolor='white')
    axes[i].set_title(f'{split.capitalize()} — {len(split_df)} tracks')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

    # Show count on each bar
    for j, v in enumerate(genre_counts.values):
        axes[i].text(j, v + 5, str(v), ha='center', fontsize=9)

plt.suptitle('Genre Distribution per Split', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

# Class weights (inverse frequency) สำหรับ CrossEntropyLoss
train_df = successful[successful['split'] == 'training']
genre_counts_train = train_df['genre_top'].value_counts().sort_index()
total_train = len(train_df)

print("\nClass Weight Analysis (Training set):")
print("-" * 55)
print(f"{'Genre':15s} | {'Count':>6s} | {'Ratio':>7s} | {'Weight':>7s}")
print("-" * 55)

class_weights = []
for genre in GENRE_LABELS:
    count = genre_counts_train.get(genre, 0)
    ratio = count / total_train
    weight = total_train / (NUM_CLASSES * count) if count > 0 else 0
    class_weights.append(weight)
    print(f"{genre:15s} | {count:6d} | {ratio:7.3f} | {weight:7.3f}")

print("-" * 55)
print(f"{'Total':15s} | {total_train:6d}")

# Imbalance ratio
max_count = genre_counts_train.max()
min_count = genre_counts_train.min()
print(f"\nImbalance ratio (max/min): {max_count/min_count:.2f}")
print(f"Most common:  {genre_counts_train.idxmax()} ({max_count})")
print(f"Least common: {genre_counts_train.idxmin()} ({min_count})")

# Save class weights
class_weights_tensor = torch.FloatTensor(class_weights)
print(f"\nClass weights tensor: {class_weights_tensor}")
print("→ ใช้ใน nn.CrossEntropyLoss(weight=class_weights_tensor) ตอน training")


## 5. 🔬 Feature Extraction & Correlation Analysis

Extract tabular features จาก sample tracks เพื่อวิเคราะห์:
- Feature correlation — ตัวไหนซ้ำซ้อน
- Feature discriminability — ตัวไหนแยก genre ได้ดี
- Outlier detection — ค่าผิดปกติ


In [ ]:
from fma_phase1a_data_pipeline import extract_tabular_features, extract_mel_spectrogram

# สุ่ม sample 500 tracks (balanced across genres) สำหรับ EDA
# ไม่ต้องใช้ทั้ง 8000 tracks — 500 ก็เห็น pattern ชัดแล้ว
SAMPLE_PER_GENRE = 60
sampled_tracks = []

for genre in GENRE_LABELS:
    genre_df = successful[successful['genre_top'] == genre]
    n_sample = min(SAMPLE_PER_GENRE, len(genre_df))
    sampled_tracks.append(genre_df.sample(n=n_sample, random_state=42))

sample_df = pd.concat(sampled_tracks).reset_index(drop=True)
print(f"Sampled {len(sample_df)} tracks for feature analysis")
print(sample_df['genre_top'].value_counts().sort_index().to_string())


In [ ]:
# Extract tabular features จาก sampled tracks
feature_names = (
    [f'mfcc_{i+1}_mean' for i in range(20)] +
    [f'mfcc_{i+1}_std' for i in range(20)] +
    ['centroid_mean', 'centroid_std'] +
    ['bandwidth_mean', 'bandwidth_std'] +
    ['rolloff_mean', 'rolloff_std'] +
    [f'chroma_{i+1}_mean' for i in range(12)] +
    [f'chroma_{i+1}_std' for i in range(12)] +
    ['zcr_mean', 'zcr_std'] +
    ['rms_mean', 'rms_std'] +
    [f'contrast_{i+1}_mean' for i in range(7)] +
    [f'contrast_{i+1}_std' for i in range(7)]
)

all_features = []
all_labels = []
all_mel_stats = []  # สำหรับ mel spectrogram analysis
extract_errors = []

print("Extracting features from samples...")
for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Feature Extraction"):
    track_id = row['track_id']
    tid_str = f"{track_id:06d}"
    audio_path = FMA_AUDIO_DIR / tid_str[:3] / f"{tid_str}.mp3"

    try:
        audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

        # Center chunk
        if len(audio) >= CHUNK_SAMPLES:
            start = (len(audio) - CHUNK_SAMPLES) // 2
            chunk = audio[start:start + CHUNK_SAMPLES]
        else:
            chunk = np.zeros(CHUNK_SAMPLES, dtype=np.float32)
            chunk[:len(audio)] = audio

        # Tabular features
        tab = extract_tabular_features(chunk, SAMPLE_RATE)
        all_features.append(tab)
        all_labels.append(row['genre_top'])

        # Mel spectrogram stats
        mel = extract_mel_spectrogram(chunk, SAMPLE_RATE)
        all_mel_stats.append({
            'mel_min': mel.min(),
            'mel_max': mel.max(),
            'mel_mean': mel.mean(),
            'mel_std': mel.std(),
        })

    except Exception as e:
        extract_errors.append((track_id, str(e)))

features_matrix = np.array(all_features)  # (N, 88)
features_df = pd.DataFrame(features_matrix, columns=feature_names)
features_df['genre'] = all_labels
mel_stats_df = pd.DataFrame(all_mel_stats)

print(f"\n✓ Extracted features from {len(features_df)} tracks")
print(f"  Errors: {len(extract_errors)}")
print(f"  Feature matrix shape: {features_matrix.shape}")
if extract_errors:
    for tid, err in extract_errors[:5]:
        print(f"  Error: Track {tid} — {err}")


### 5.1 Feature Correlation Heatmap

In [ ]:
# Correlation matrix ของ tabular features
corr_matrix = features_df[feature_names].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.1, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'shrink': 0.6, 'label': 'Correlation'})
ax.set_title('Feature Correlation Heatmap (88 features)', fontsize=14)
ax.tick_params(labelsize=5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Highly correlated pairs (|r| > 0.9)
high_corr = []
for i in range(len(feature_names)):
    for j in range(i+1, len(feature_names)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr.append((feature_names[i], feature_names[j], r))

print(f"\nHighly correlated feature pairs (|r| > 0.9): {len(high_corr)}")
for f1, f2, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)[:15]:
    print(f"  {r:+.3f} | {f1:25s} ↔ {f2}")


### 5.2 Feature Discriminability (ANOVA F-score)

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder

# ANOVA F-test: วัดว่า feature แต่ละตัวแยก genre ได้ดีแค่ไหน
X = features_df[feature_names].values
y = LabelEncoder().fit_transform(features_df['genre'].values)

f_scores, p_values = f_classif(X, y)

# Sort by F-score
f_score_df = pd.DataFrame({
    'feature': feature_names,
    'f_score': f_scores,
    'p_value': p_values,
}).sort_values('f_score', ascending=False)

# Plot top 30
fig, ax = plt.subplots(figsize=(12, 8))
top_n = 30
top_features = f_score_df.head(top_n)
colors = ['#4CAF50' if p < 0.001 else '#FFC107' if p < 0.05 else '#F44336'
          for p in top_features['p_value']]
ax.barh(range(top_n), top_features['f_score'].values, color=colors, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values, fontsize=9)
ax.set_xlabel('ANOVA F-Score')
ax.set_title(f'Top {top_n} Most Discriminative Features (Green=p<0.001, Yellow=p<0.05, Red=p≥0.05)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 10 most discriminative features:")
print(f"{'Rank':>4s} | {'Feature':25s} | {'F-Score':>10s} | {'p-value':>12s}")
print("-" * 60)
for i, (_, row) in enumerate(f_score_df.head(10).iterrows()):
    sig = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else "ns"
    print(f"{i+1:4d} | {row['feature']:25s} | {row['f_score']:10.1f} | {row['p_value']:.2e} {sig}")

print(f"\nLeast discriminative features:")
for i, (_, row) in enumerate(f_score_df.tail(5).iterrows()):
    print(f"  {row['feature']:25s} | F={row['f_score']:.1f} | p={row['p_value']:.2e}")


### 5.3 Feature Distribution by Genre (Top Features)

In [ ]:
# Plot distributions ของ top 6 features แยกตาม genre
top6_features = f_score_df.head(6)['feature'].values

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top6_features):
    ax = axes[i]
    for genre in GENRE_LABELS:
        vals = features_df[features_df['genre'] == genre][feat]
        ax.hist(vals, bins=25, alpha=0.4, label=genre, density=True)
    ax.set_title(f'{feat}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Top 6 Discriminative Features — Distribution by Genre', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_top_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.4 Outlier Detection in Tabular Features

In [ ]:
# ตรวจ outliers ด้วย IQR method
outlier_counts = {}

for feat in feature_names:
    vals = features_df[feat]
    Q1 = vals.quantile(0.25)
    Q3 = vals.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3.0 * IQR  # ใช้ 3x IQR (extreme outliers เท่านั้น)
    upper = Q3 + 3.0 * IQR
    n_outliers = ((vals < lower) | (vals > upper)).sum()
    if n_outliers > 0:
        outlier_counts[feat] = n_outliers

print(f"Features with extreme outliers (3x IQR):")
print(f"{'Feature':30s} | {'# Outliers':>10s} | {'% of Data':>10s}")
print("-" * 55)
for feat, count in sorted(outlier_counts.items(), key=lambda x: x[1], reverse=True)[:15]:
    pct = count / len(features_df) * 100
    print(f"{feat:30s} | {count:10d} | {pct:9.1f}%")

print(f"\nTotal features with outliers: {len(outlier_counts)} / {len(feature_names)}")
print("\n→ สำหรับ audio features, outliers เป็นเรื่องปกติ (เพลงมี variance สูง)")
print("→ ไม่ควรตัดทิ้ง — ใช้ RobustScaler หรือ BatchNorm แทน")

# Check for NaN / Inf in features
nan_count = features_df[feature_names].isna().sum().sum()
inf_count = np.isinf(features_matrix).sum()
print(f"\n✓ NaN values in features: {nan_count}")
print(f"✓ Inf values in features: {inf_count}")


## 6. 📊 Mel Spectrogram Statistics & Normalization Strategy

วิเคราะห์ dB value distribution เพื่อเลือก normalization strategy ที่ดีที่สุด


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# dB value distribution
axes[0].hist(mel_stats_df['mel_mean'], bins=40, color='#E91E63', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Mean dB Value')
axes[0].set_ylabel('Count')
axes[0].set_title('Mel Spectrogram Mean dB Distribution')

axes[1].hist(mel_stats_df['mel_std'], bins=40, color='#3F51B5', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Std dB Value')
axes[1].set_ylabel('Count')
axes[1].set_title('Mel Spectrogram Std dB Distribution')

# Min/Max range
axes[2].scatter(mel_stats_df['mel_min'], mel_stats_df['mel_max'],
                alpha=0.3, s=10, color='#009688')
axes[2].set_xlabel('Min dB')
axes[2].set_ylabel('Max dB')
axes[2].set_title('Mel Spectrogram dB Range per Track')
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='0 dB (max ref)')
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_mel_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mel Spectrogram Statistics:")
print(f"  Mean dB — mean: {mel_stats_df['mel_mean'].mean():.2f}, std: {mel_stats_df['mel_mean'].std():.2f}")
print(f"  Std dB  — mean: {mel_stats_df['mel_std'].mean():.2f}, std: {mel_stats_df['mel_std'].std():.2f}")
print(f"  Min dB  — mean: {mel_stats_df['mel_min'].mean():.2f}")
print(f"  Max dB  — always: {mel_stats_df['mel_max'].max():.2f} (ref=np.max → 0 dB)")

# Compute global mean & std สำหรับ normalization
global_mel_mean = mel_stats_df['mel_mean'].mean()
global_mel_std = mel_stats_df['mel_std'].mean()
print(f"\n→ Recommended Global Normalization:")
print(f"  mean = {global_mel_mean:.4f}")
print(f"  std  = {global_mel_std:.4f}")
print(f"  normalize: (mel - {global_mel_mean:.2f}) / {global_mel_std:.2f}")
print(f"\n→ Alternative: Per-sample normalization (แต่ละ spectrogram normalize ด้วย mean/std ของตัวเอง)")
print(f"→ จะเปรียบเทียบทั้ง 2 วิธีตอน training phase")


## 7. 🎶 Per-Genre Audio Comparison

In [ ]:
# โหลด 1 ตัวอย่างจากแต่ละ genre แล้วเปรียบเทียบ spectrogram
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 4, hspace=0.4, wspace=0.3)

for i, genre in enumerate(GENRE_LABELS):
    genre_tracks = successful[successful['genre_top'] == genre]
    sample_track = genre_tracks.iloc[0]
    track_id = sample_track['track_id']
    tid_str = f"{track_id:06d}"
    audio_path = FMA_AUDIO_DIR / tid_str[:3] / f"{tid_str}.mp3"

    try:
        audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        start = (len(audio) - CHUNK_SAMPLES) // 2
        chunk = audio[start:start + CHUNK_SAMPLES]
        mel = extract_mel_spectrogram(chunk, SAMPLE_RATE)

        ax = fig.add_subplot(gs[i // 4, i % 4])
        ax.imshow(mel[0], aspect='auto', origin='lower', cmap='magma',
                  extent=[0, CHUNK_DURATION, 0, SAMPLE_RATE // 2])
        ax.set_title(f'{genre}\n(Track {track_id})', fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.set_ylabel('Freq (Hz)', fontsize=8)
        ax.tick_params(labelsize=7)
    except Exception as e:
        ax = fig.add_subplot(gs[i // 4, i % 4])
        ax.text(0.5, 0.5, f'Error\n{e}', ha='center', va='center')

plt.suptitle('Mel Spectrograms by Genre (1 sample each)', fontsize=14, y=1.02)
plt.savefig(OUTPUT_DIR / 'eda_genre_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. 🧹 Cleaning Summary & Export

รวม findings ทั้งหมดแล้วสร้าง cleaned metadata สำหรับ Phase 1B


In [ ]:
print("=" * 70)
print("  CLEANING SUMMARY")
print("=" * 70)

# Collect all tracks to remove
tracks_to_remove = set()
removal_reasons = {}

# 1. Failed to load
failed_tracks = set(scan_df[~scan_df['load_success']]['track_id'].values)
for tid in failed_tracks:
    tracks_to_remove.add(tid)
    removal_reasons[tid] = 'load_failed'
print(f"\n1. Load failures:         {len(failed_tracks)} tracks")

# 2. Silent tracks
silent_tids = set(scan_df[scan_df['rms_mean'] < 0.001]['track_id'].values)
for tid in silent_tids:
    tracks_to_remove.add(tid)
    removal_reasons[tid] = 'silent'
print(f"2. Silent (RMS<0.001):    {len(silent_tids)} tracks")

# 3. Very short tracks (< 1 second — too short to extract meaningful features)
very_short = set(scan_df[(scan_df['load_success']) & (scan_df['duration_actual'] < 1.0)]['track_id'].values)
for tid in very_short:
    tracks_to_remove.add(tid)
    removal_reasons[tid] = 'too_short'
print(f"3. Too short (<1s):       {len(very_short)} tracks")

# 4. Missing files
missing_files = set(scan_df[~scan_df['file_exists']]['track_id'].values)
for tid in missing_files:
    tracks_to_remove.add(tid)
    removal_reasons[tid] = 'file_missing'
print(f"4. File missing:          {len(missing_files)} tracks")

# 5. Original corrupted list (already removed, but confirm)
print(f"5. Original corrupted:    {len(CORRUPTED_TRACK_IDS)} tracks (already excluded)")

print(f"\n{'─' * 40}")
print(f"Total unique tracks to remove: {len(tracks_to_remove)}")

# Create cleaned metadata
clean_df = successful[~successful['track_id'].isin(tracks_to_remove)].copy()

print(f"\nBefore cleaning: {len(successful)} tracks")
print(f"After cleaning:  {len(clean_df)} tracks")
print(f"Removed:         {len(successful) - len(clean_df)} tracks")

# Updated genre distribution
print(f"\nCleaned genre distribution:")
for genre in GENRE_LABELS:
    count = len(clean_df[clean_df['genre_top'] == genre])
    train_count = len(clean_df[(clean_df['genre_top'] == genre) & (clean_df['split'] == 'training')])
    print(f"  {genre:15s}: {count:5d} total ({train_count:4d} train)")

# Updated skip-list
UPDATED_SKIP_LIST = CORRUPTED_TRACK_IDS | tracks_to_remove
print(f"\n✓ Updated skip-list: {len(UPDATED_SKIP_LIST)} track IDs")
print(f"  {sorted(UPDATED_SKIP_LIST)[:20]}{'...' if len(UPDATED_SKIP_LIST) > 20 else ''}")


In [ ]:
# Save cleaned metadata
clean_export = clean_df[['track_id', 'genre_top', 'genre_idx', 'split',
                          'duration_actual', 'rms_mean', 'file_size_kb']].copy()
clean_export.to_csv(OUTPUT_DIR / 'cleaned_metadata.csv', index=False)
print(f"✓ Saved cleaned metadata to: {OUTPUT_DIR / 'cleaned_metadata.csv'}")
print(f"  Shape: {clean_export.shape}")

# Save updated skip-list
skip_list_path = OUTPUT_DIR / 'skip_track_ids.txt'
with open(skip_list_path, 'w') as f:
    for tid in sorted(UPDATED_SKIP_LIST):
        f.write(f"{tid}\n")
print(f"✓ Saved skip-list to: {skip_list_path}")

# Save normalization stats
norm_stats = {
    'global_mel_mean': float(global_mel_mean),
    'global_mel_std': float(global_mel_std),
    'class_weights': class_weights,
}
import json
with open(OUTPUT_DIR / 'normalization_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)
print(f"✓ Saved normalization stats to: {OUTPUT_DIR / 'normalization_stats.json'}")

print(f"\n{'=' * 70}")
print(f"  EDA COMPLETE — Ready for Phase 1B (Pre-extract .pt files)")
print(f"{'=' * 70}")


## 9. 📋 Key Findings & Recommendations

### Findings จะปรากฏหลังรัน notebook — สรุปสิ่งที่ต้องดู:

| Check | ดูอะไร | Action |
|-------|--------|--------|
| Duration | เพลงส่วนใหญ่ ~30s — 3s chunk เหมาะสม | ถ้ามีเพลง <1s ให้ remove |
| Class Imbalance | ratio max/min genre | ถ้า >2x ใช้ class_weights ใน loss |
| Silent Tracks | RMS ≈ 0 | Remove — features เป็น garbage |
| Feature Correlation | MFCC mean/std มัก correlate สูง | ปกติ, ไม่ต้อง drop — ให้ model เลือกเอง |
| Top Features | MFCCs มักสำคัญสุด | ยืนยัน tabular baseline จะ work |
| Mel Normalization | dB range ~-80 to 0 | ใช้ global mean/std หรือ per-sample |
| Outliers | Audio มี outliers ตามธรรมชาติ | ใช้ BatchNorm, ไม่ตัดทิ้ง |

### Files ที่ได้จาก EDA:
- `output/cleaned_metadata.csv` — metadata ที่ clean แล้ว
- `output/skip_track_ids.txt` — updated skip-list
- `output/normalization_stats.json` — global mel mean/std + class weights
- `output/eda_*.png` — visualization plots ทั้งหมด
